In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("CICIDS2017_ML") \
    .master("local[3]") \
    .config("spark.driver.memory", "10g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.memory.offHeap.enabled", "true") \
    .config("spark.memory.offHeap.size", "2g") \
    .getOrCreate()



Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/09 19:51:42 WARN Utils: Your hostname, aswin-Inspiron, resolves to a loopback address: 127.0.1.1; using 192.168.26.189 instead (on interface wlp1s0)
26/03/09 19:51:42 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/09 19:51:44 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Load the cleaned Parquet data

In [2]:
# Load from Parquet (much faster than re-reading CSVs)
df = spark.read.parquet("./output/cleaned/")
print(f"Rows: {df.count():,}")
print(f"Columns: {len(df.columns)}")

[Stage 1:===============================================>           (4 + 1) / 5]

Rows: 2,564,722
Columns: 85


Select features

In [3]:
from pyspark.sql.types import NumericType

# Exclude non-feature columns
exclude_cols = [
    "Label", "label_index", "is_attack",
    "source_file", "duration_bucket",
    "features_raw", "features"        # if still present
]

feature_cols = [
    f.name for f in df.schema.fields
    if isinstance(f.dataType, NumericType)
    and f.name not in exclude_cols
]

print(f"Features selected: {len(feature_cols)}")
print(feature_cols[:10])  # preview first 10

Features selected: 80
['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std']


### 1.3 Assemble features into ML vector

In [4]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features",
    handleInvalid="skip"
)
df_ml = assembler.transform(df)

# Keep only what ML needs
df_ml = df_ml.select("features", "is_attack", "label_index", "Label")
df_ml.show(3, truncate=True)

26/03/09 19:52:02 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
[Stage 4:>                                                          (0 + 1) / 1]

+--------------------+---------+-----------+------+
|            features|is_attack|label_index| Label|
+--------------------+---------+-----------+------+
|[80.0,1.30945535E...|        0|        0.0|BENIGN|
|(80,[0,1,2,3,4,5,...|        0|        0.0|BENIGN|
|(80,[0,1,2,3,4,5,...|        0|        0.0|BENIGN|
+--------------------+---------+-----------+------+
only showing top 3 rows


#### 1.4 Handle class imbalance


In [5]:
from pyspark.sql.functions import col

# Check distribution
df_ml.groupBy("is_attack").count().show()
# is_attack=0 : 2,138,844 (83.4%)
# is_attack=1 :   425,878 (16.6%)

# Calculate class weights to compensate imbalance
total    = df_ml.count()
benign   = df_ml.filter(col("is_attack") == 0).count()
attack   = df_ml.filter(col("is_attack") == 1).count()

weight_benign = total / (2.0 * benign)
weight_attack = total / (2.0 * attack)

print(f"Weight for BENIGN : {weight_benign:.4f}")
print(f"Weight for ATTACK : {weight_attack:.4f}")

# Add weight column
from pyspark.sql.functions import when
df_ml = df_ml.withColumn("classWeight",
    when(col("is_attack") == 0, weight_benign)
    .otherwise(weight_attack))

+---------+-------+
|is_attack|  count|
+---------+-------+
|        0|2138844|
|        1| 425878|
+---------+-------+



[Stage 14:===========================================>              (3 + 1) / 4]

Weight for BENIGN : 0.5996
Weight for ATTACK : 3.0111


#### Train/Test Split


In [6]:
# 80% train, 20% test — stratified by is_attack
train_df, test_df = df_ml.randomSplit([0.8, 0.2], seed=42)

# Verify split is balanced
print(f"Train rows : {train_df.count():,}")
print(f"Test rows  : {test_df.count():,}")

# Check class distribution in train set
train_df.groupBy("is_attack").count().show()
test_df.groupBy("is_attack").count().show()

Train rows : 2,051,452


Test rows  : 513,270


+---------+-------+
|is_attack|  count|
+---------+-------+
|        0|1710878|
|        1| 340574|
+---------+-------+



[Stage 26:==============================================>           (4 + 1) / 5]

+---------+------+
|is_attack| count|
+---------+------+
|        0|427966|
|        1| 85304|
+---------+------+



#  Building a Machine Learning Model

### Model 1 — Random Forest

In [7]:
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="is_attack",
    weightCol="classWeight",
    numTrees=50,           # start with 50, tune later
    maxDepth=10,
    maxBins=32,
    seed=42
)

rf_model = rf.fit(train_df)
rf_predictions = rf_model.transform(test_df)
rf_predictions.select("Label", "is_attack", "prediction", "probability").show(10)

26/03/09 20:05:09 WARN DAGScheduler: Broadcasting large task binary with size 1324.2 KiB
26/03/09 20:05:32 WARN DAGScheduler: Broadcasting large task binary with size 1865.8 KiB
26/03/09 20:05:58 WARN DAGScheduler: Broadcasting large task binary with size 2.5 MiB
26/03/09 20:06:24 WARN DAGScheduler: Broadcasting large task binary with size 1758.0 KiB
[Stage 56:>                                                         (0 + 1) / 1]

+------+---------+----------+--------------------+
| Label|is_attack|prediction|         probability|
+------+---------+----------+--------------------+
|BENIGN|        0|       0.0|[0.96979957470169...|
|BENIGN|        0|       0.0|[0.97594927440471...|
|BENIGN|        0|       0.0|[0.98119820257125...|
|BENIGN|        0|       0.0|[0.99634060098009...|
|BENIGN|        0|       0.0|[0.99060120046598...|
|BENIGN|        0|       0.0|[0.98914856124352...|
|BENIGN|        0|       0.0|[0.99731543888294...|
|BENIGN|        0|       0.0|[0.99727791689662...|
|BENIGN|        0|       0.0|[0.99640937208978...|
|BENIGN|        0|       0.0|[0.99640937208978...|
+------+---------+----------+--------------------+
only showing top 10 rows


### Model 2 — Logistic Regression

In [8]:
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(
    featuresCol="features",
    labelCol="is_attack",
    weightCol="classWeight",
    maxIter=20,
    regParam=0.01,
    elasticNetParam=0.0
)

lr_model = lr.fit(train_df)
lr_predictions = lr_model.transform(test_df)

### Model 3 — Decision Tree

In [9]:
from pyspark.ml.classification import DecisionTreeClassifier

dt = DecisionTreeClassifier(
    featuresCol="features",
    labelCol="is_attack",
    weightCol="classWeight",
    maxDepth=10,
    maxBins=32,
    seed=42
)

dt_model = dt.fit(train_df)
dt_predictions = dt_model.transform(test_df)

### Model Evaluation

In [10]:
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator
)

def evaluate_model(predictions, model_name):
    print(f"\n{'='*50}")
    print(f"  {model_name}")
    print(f"{'='*50}")

    # Accuracy
    acc_eval = MulticlassClassificationEvaluator(
        labelCol="is_attack",
        predictionCol="prediction",
        metricName="accuracy"
    )
    accuracy = acc_eval.evaluate(predictions)

    # Precision
    prec_eval = MulticlassClassificationEvaluator(
        labelCol="is_attack",
        predictionCol="prediction",
        metricName="weightedPrecision"
    )
    precision = prec_eval.evaluate(predictions)

    # Recall
    rec_eval = MulticlassClassificationEvaluator(
        labelCol="is_attack",
        predictionCol="prediction",
        metricName="weightedRecall"
    )
    recall = rec_eval.evaluate(predictions)

    # F1 Score
    f1_eval = MulticlassClassificationEvaluator(
        labelCol="is_attack",
        predictionCol="prediction",
        metricName="f1"
    )
    f1 = f1_eval.evaluate(predictions)

    # AUC-ROC
    auc_eval = BinaryClassificationEvaluator(
        labelCol="is_attack",
        rawPredictionCol="rawPrediction",
        metricName="areaUnderROC"
    )
    auc = auc_eval.evaluate(predictions)

    print(f"  Accuracy  : {accuracy:.4f}  ({accuracy*100:.2f}%)")
    print(f"  Precision : {precision:.4f}")
    print(f"  Recall    : {recall:.4f}")
    print(f"  F1 Score  : {f1:.4f}")
    print(f"  AUC-ROC   : {auc:.4f}")

    return {
        "Model": model_name,
        "Accuracy": round(accuracy, 4),
        "Precision": round(precision, 4),
        "Recall": round(recall, 4),
        "F1": round(f1, 4),
        "AUC-ROC": round(auc, 4)
    }

In [11]:
results = []
results.append(evaluate_model(rf_predictions, "Random Forest"))
results.append(evaluate_model(lr_predictions, "Logistic Regression"))
results.append(evaluate_model(dt_predictions, "Decision Tree"))


  Random Forest


26/03/09 20:14:53 WARN DAGScheduler: Broadcasting large task binary with size 1768.6 KiB
26/03/09 20:16:08 WARN DAGScheduler: Broadcasting large task binary with size 1768.6 KiB
26/03/09 20:17:21 WARN DAGScheduler: Broadcasting large task binary with size 1768.6 KiB
26/03/09 20:18:31 WARN DAGScheduler: Broadcasting large task binary with size 1768.6 KiB
26/03/09 20:19:41 WARN DAGScheduler: Broadcasting large task binary with size 1755.6 KiB
                                                                                

  Accuracy  : 0.9974  (99.74%)
  Precision : 0.9974
  Recall    : 0.9974
  F1 Score  : 0.9974
  AUC-ROC   : 0.9998

  Logistic Regression


  Accuracy  : 0.8898  (88.98%)
  Precision : 0.9271
  Recall    : 0.8898
  F1 Score  : 0.8988
  AUC-ROC   : 0.9762

  Decision Tree


[Stage 174:=============================================>           (4 + 1) / 5]

  Accuracy  : 0.9961  (99.61%)
  Precision : 0.9961
  Recall    : 0.9961
  F1 Score  : 0.9961
  AUC-ROC   : 0.9971


#### Confusion matrix

In [12]:
from pyspark.sql.functions import col

def confusion_matrix(predictions, model_name):
    print(f"\nConfusion Matrix — {model_name}")
    predictions.groupBy("is_attack", "prediction") \
               .count() \
               .orderBy("is_attack", "prediction") \
               .show()

confusion_matrix(rf_predictions, "Random Forest")


Confusion Matrix — Random Forest


26/03/09 20:32:25 WARN DAGScheduler: Broadcasting large task binary with size 1766.7 KiB
[Stage 185:=============================================>           (4 + 1) / 5]

+---------+----------+------+
|is_attack|prediction| count|
+---------+----------+------+
|        0|       0.0|427297|
|        0|       1.0|   669|
|        1|       0.0|   642|
|        1|       1.0| 84662|
+---------+----------+------+



26/03/09 20:33:38 WARN DAGScheduler: Broadcasting large task binary with size 1661.3 KiB
                                                                                

## Compare all models


In [13]:
import pandas as pd

results_df = pd.DataFrame(results)
results_df = results_df.set_index("Model")
print("\n===== MODEL COMPARISON =====")
print(results_df.to_string())


===== MODEL COMPARISON =====
                     Accuracy  Precision  Recall      F1  AUC-ROC
Model                                                            
Random Forest          0.9974     0.9974  0.9974  0.9974   0.9998
Logistic Regression    0.8898     0.9271  0.8898  0.8988   0.9762
Decision Tree          0.9961     0.9961  0.9961  0.9961   0.9971


###  Feature importance (Random Forest)


In [14]:
import pandas as pd

importances = rf_model.featureImportances
feature_importance_df = pd.DataFrame({
    "Feature": feature_cols,
    "Importance": list(importances)
}).sort_values("Importance", ascending=False)

print("\nTop 15 Most Important Features:")
print(feature_importance_df.head(15).to_string(index=False))


Top 15 Most Important Features:
                    Feature  Importance
      Bwd Packet Length Std    0.078787
     Packet Length Variance    0.078720
          Max Packet Length    0.077657
     Bwd Packet Length Mean    0.063901
        Average Packet Size    0.054320
          Subflow Bwd Bytes    0.043620
      Bwd Packet Length Max    0.037407
      Fwd Packet Length Max    0.037050
         Packet Length Mean    0.035471
          Packet Length Std    0.035272
           Destination Port    0.033884
Total Length of Fwd Packets    0.032559
         Fwd Header Length2    0.021529
     Fwd Packet Length Mean    0.019629
       Avg Fwd Segment Size    0.019145


#  Hyperparameter Tuning
### Cross Validation with Grid Search

In [15]:
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# Use Random Forest
rf_tuning = RandomForestClassifier(
    featuresCol="features",
    labelCol="is_attack",
    weightCol="classWeight",
    seed=42
)

# Define parameter grid
paramGrid = ParamGridBuilder() \
    .addGrid(rf_tuning.numTrees,  [20, 50, 100]) \
    .addGrid(rf_tuning.maxDepth,  [5, 10, 15]) \
    .addGrid(rf_tuning.maxBins,   [16, 32]) \
    .build()

print(f"Total parameter combinations: {len(paramGrid)}")
# 3 x 3 x 2 = 18 combinations

Total parameter combinations: 18


In [16]:
# Use a sample for tuning, then train best params on full data
sample_train = train_df.sample(fraction=0.2, seed=42)
print(f"Tuning on sample: {sample_train.count():,} rows")

evaluator = BinaryClassificationEvaluator(
    labelCol="is_attack",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

cv = CrossValidator(
    estimator=rf_tuning,
    estimatorParamMaps=paramGrid,
    evaluator=evaluator,
    numFolds=3,           # 3-fold CV
    seed=42
)

cv_model = cv.fit(sample_train)
print("Cross validation complete!")

Tuning on sample: 411,083 rows


26/03/09 20:38:16 WARN DAGScheduler: Broadcasting large task binary with size 1097.8 KiB
26/03/09 20:38:17 WARN DAGScheduler: Broadcasting large task binary with size 1299.6 KiB
26/03/09 20:38:19 WARN DAGScheduler: Broadcasting large task binary with size 1502.5 KiB
26/03/09 20:38:21 WARN DAGScheduler: Broadcasting large task binary with size 1686.0 KiB
26/03/09 20:38:22 WARN DAGScheduler: Broadcasting large task binary with size 1841.5 KiB
26/03/09 20:38:24 WARN DAGScheduler: Broadcasting large task binary with size 1146.2 KiB
26/03/09 20:38:44 WARN DAGScheduler: Broadcasting large task binary with size 1083.0 KiB
26/03/09 20:38:46 WARN DAGScheduler: Broadcasting large task binary with size 1279.9 KiB
26/03/09 20:38:47 WARN DAGScheduler: Broadcasting large task binary with size 1467.2 KiB
26/03/09 20:38:49 WARN DAGScheduler: Broadcasting large task binary with size 1641.2 KiB
26/03/09 20:38:51 WARN DAGScheduler: Broadcasting large task binary with size 1798.3 KiB
26/03/09 20:38:53 WAR

Cross validation complete!


####  Best parameters

In [17]:
best_rf = cv_model.bestModel

print("Best Parameters:")
print(f"  numTrees : {best_rf.getNumTrees}")
print(f"  maxDepth : {best_rf.getOrDefault('maxDepth')}")
print(f"  maxBins  : {best_rf.getOrDefault('maxBins')}")

# CV scores per fold
avg_metrics = cv_model.avgMetrics
print(f"\nCV AUC-ROC scores: {[round(m,4) for m in avg_metrics]}")
print(f"Best AUC-ROC     : {max(avg_metrics):.4f}")

Best Parameters:
  numTrees : 100
  maxDepth : 15
  maxBins  : 32

CV AUC-ROC scores: [np.float64(0.9954), np.float64(0.9964), np.float64(0.9996), np.float64(0.9997), np.float64(0.9999), np.float64(0.9999), np.float64(0.9964), np.float64(0.9969), np.float64(0.9997), np.float64(0.9997), np.float64(0.9999), np.float64(0.9999), np.float64(0.9964), np.float64(0.9971), np.float64(0.9997), np.float64(0.9997), np.float64(0.9999), np.float64(0.9999)]
Best AUC-ROC     : 0.9999


### Train final model with best parameters on full data

In [18]:
# Train best config on full training set
final_rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="is_attack",
    weightCol="classWeight",
    numTrees=best_rf.getNumTrees,
    maxDepth=best_rf.getOrDefault("maxDepth"),
    maxBins=best_rf.getOrDefault("maxBins"),
    seed=42
)

final_model = final_rf.fit(train_df)
final_predictions = final_model.transform(test_df)

# Final evaluation
evaluate_model(final_predictions, "Random Forest (Tuned)")

26/03/09 21:41:33 WARN DAGScheduler: Broadcasting large task binary with size 1036.2 KiB
26/03/09 21:42:21 WARN DAGScheduler: Broadcasting large task binary with size 1566.3 KiB
26/03/09 21:43:08 WARN DAGScheduler: Broadcasting large task binary with size 2.3 MiB
26/03/09 21:43:58 WARN DAGScheduler: Broadcasting large task binary with size 3.3 MiB
26/03/09 21:44:55 WARN DAGScheduler: Broadcasting large task binary with size 4.6 MiB
26/03/09 21:45:55 WARN DAGScheduler: Broadcasting large task binary with size 6.1 MiB
26/03/09 21:46:54 WARN DAGScheduler: Broadcasting large task binary with size 7.8 MiB
26/03/09 21:47:57 WARN DAGScheduler: Broadcasting large task binary with size 9.5 MiB
26/03/09 21:49:17 WARN DAGScheduler: Broadcasting large task binary with size 11.3 MiB
26/03/09 21:50:46 WARN DAGScheduler: Broadcasting large task binary with size 13.0 MiB
                                                                                


  Random Forest (Tuned)


26/03/09 21:52:09 WARN DAGScheduler: Broadcasting large task binary with size 7.4 MiB
26/03/09 21:53:43 WARN DAGScheduler: Broadcasting large task binary with size 7.4 MiB
26/03/09 21:55:18 WARN DAGScheduler: Broadcasting large task binary with size 7.4 MiB
26/03/09 21:56:51 WARN DAGScheduler: Broadcasting large task binary with size 7.4 MiB
26/03/09 21:58:31 WARN DAGScheduler: Broadcasting large task binary with size 7.3 MiB
                                                                                

  Accuracy  : 0.9985  (99.85%)
  Precision : 0.9985
  Recall    : 0.9985
  F1 Score  : 0.9985
  AUC-ROC   : 0.9999


{'Model': 'Random Forest (Tuned)',
 'Accuracy': 0.9985,
 'Precision': 0.9985,
 'Recall': 0.9985,
 'F1': 0.9985,
 'AUC-ROC': 0.9999}

### Save the final model

In [19]:
# Save model to disk
final_model.save("./output/models/random_forest_final/")
print("Model saved!")

# Load it back later
from pyspark.ml.classification import RandomForestClassificationModel
loaded_model = RandomForestClassificationModel.load("./output/models/random_forest_final/")
print("Model loaded successfully!")

26/03/09 22:00:08 WARN TaskSetManager: Stage 2343 contains a task of very large size (6591 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

Model saved!


Model loaded successfully!
